# ECG Heartbeat Arrhythmia Classification

Classify single heartbeats from hospital ECG recordings into **four arrhythmia categories** (labels `0`, `1`, `2`, `3`).

**The data**
- Every row of `train.csv` / `test.csv` is one heartbeat.
- `sig_0 ... sig_249`: 250 ECG signal values centred on the heartbeat.
- `pre_rr`, `post_rr`, `rr_ratio`: three timing features. They describe the gap between this heartbeat and its neighbours (the RR interval before, the RR interval after, and the ratio of the two).
- `label`: the category (only in `train.csv`).

**The metric**

Submissions are scored with **Macro F1**: the F1 score is calculated separately for each of the four classes and then averaged, so every class counts equally. This matters because the classes are very unbalanced (class 3 has only about 600 training beats, class 0 has 45,000).

**Approach**
1. Load the data and fix missing values.
2. Build a small 1D convolutional neural network that reads the ECG signal, plus a small dense network that reads the three timing features. Their outputs are combined to make the final prediction.
3. Handle class imbalance with a weighted sampler and a weighted loss.
4. Train with 5-fold cross-validation, so every training beat is also used once as unseen validation data.
5. Predict the test set by averaging the probabilities of the 5 fold models.

**Notebook outline**
1. Imports and settings
2. Load and clean the training data
3. Dataset class (with data augmentation)
4. Model
5. Evaluation helpers and the single-fold training function
6. Cross-validation training
7. Out-of-fold results
8. Save models
9. Predict the test set
10. Write and check `submission.csv`

## 1. Imports and settings

All adjustable settings are at the top of the cell below. The two you are most likely to change are `NUMBER_OF_FOLDS` and `MAX_EPOCHS`. Lower values give a faster but slightly noisier result.

In [ ]:
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as functional
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DATA_DIR = "/kaggle/input/competitions/nppe-2-t-2-26-ecg-heartbeat-arrhythmia-classification/nppe2_dataset"

NUMBER_OF_CLASSES = 4
NUMBER_OF_FOLDS = 5         # use 3 for a faster (slightly noisier) run
MAX_EPOCHS = 20             # maximum training epochs per fold
EARLY_STOP_PATIENCE = 5     # stop a fold after this many epochs without improvement
SMOOTHING_WINDOW = 3        # validation F1 is averaged over this many recent epochs

# Use the GPU if one is available, otherwise the CPU.
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print("Using device:", device)

## 2. Load and clean the training data

The columns are split into two groups:
- **Signal columns** (`sig_...`): the 250 ECG values of each heartbeat.
- **Timing columns** (`pre_rr`, `post_rr`, `rr_ratio`): the three RR-interval features.

The `value_counts()` output at the bottom shows how unbalanced the classes are.

In [ ]:
start_time = time.time()
train_data = pd.read_csv(f"{DATA_DIR}/train.csv")
load_seconds = time.time() - start_time
print("train shape:", train_data.shape, f"({load_seconds:.1f}s)")

signal_columns = []
for column_name in train_data.columns:
    if column_name.startswith("sig_"):
        signal_columns.append(column_name)

rr_interval_columns = ["pre_rr", "post_rr", "rr_ratio"]
print("n signal columns:", len(signal_columns))

# How many heartbeats belong to each class?
train_data["label"].value_counts()

### Fixing missing and extreme values

- A few timing values are missing (`NaN`). They are replaced with the **median** of their column. The medians are calculated on the training data only, and the same medians are reused later for the test data.
- Signal values are clipped to the range -10 to 10 so that rare extreme spikes cannot dominate training.

In [ ]:
def fill_missing_with_medians(features, column_medians):
    """Replace every NaN in `features` with the median of its column."""
    for column_index in range(features.shape[1]):
        is_missing = np.isnan(features[:, column_index])
        features[is_missing, column_index] = column_medians[column_index]
    return features


all_signals = train_data[signal_columns].values.astype(np.float32)
all_rr_features = train_data[rr_interval_columns].values.astype(np.float32)
all_labels = train_data["label"].values.astype(np.int64)

# Medians are computed before filling, so they ignore the missing values.
column_medians = np.nanmedian(all_rr_features, axis=0)

missing_value_count = np.isnan(all_rr_features).sum()
if missing_value_count > 0:
    print(f"Imputing {missing_value_count} NaN values in extra features with medians {column_medians}")
    all_rr_features = fill_missing_with_medians(all_rr_features, column_medians)

all_signals = np.clip(all_signals, -10, 10)
print("Any remaining NaNs:", np.isnan(all_rr_features).any(), np.isnan(all_signals).any())

## 3. Dataset class with data augmentation

`ECGDataset` returns one heartbeat at a time as PyTorch tensors.

When `use_augmentation=True` (training only), each signal is randomly changed a little, so the model sees slightly different versions of each beat every epoch. Each change is applied with a 50% chance:
- **Noise**: add a small amount of random noise.
- **Scaling**: multiply the signal by a factor between 0.9 and 1.1.
- **Time shift**: move the signal left or right by up to 8 samples.

In [ ]:
class ECGDataset(Dataset):
    def __init__(self, signals, rr_features, labels=None, use_augmentation=False):
        self.signals = signals
        self.rr_features = rr_features
        self.labels = labels                  # None when predicting the test set
        self.use_augmentation = use_augmentation

    def __len__(self):
        return len(self.signals)

    def __getitem__(self, index):
        signal = self.signals[index].copy()
        rr_features = self.rr_features[index]

        if self.use_augmentation:
            # 1. Add small random noise
            if np.random.rand() < 0.5:
                noise = np.random.normal(0, 0.03, size=signal.shape).astype(np.float32)
                signal = signal + noise

            # 2. Randomly make the whole signal a little larger or smaller
            if np.random.rand() < 0.5:
                scale_factor = np.random.uniform(0.9, 1.1)
                signal = signal * scale_factor

            # 3. Shift the signal by a few samples (values that fall off one end wrap around)
            if np.random.rand() < 0.5:
                shift_amount = np.random.randint(-8, 9)
                signal = np.roll(signal, shift_amount)

        # Add a channel dimension: (250,) -> (1, 250), as required by Conv1d
        signal_tensor = torch.from_numpy(signal).unsqueeze(0)
        rr_tensor = torch.from_numpy(rr_features)

        if self.labels is not None:
            label_tensor = torch.tensor(self.labels[index], dtype=torch.long)
            return signal_tensor, rr_tensor, label_tensor
        return signal_tensor, rr_tensor

## 4. Model

The network has two branches whose outputs are joined before the final decision:

1. **Signal branch**: a 1D convolutional network with three residual blocks. Each residual block adds its input back to its output, which keeps training stable. Max-pooling halves the signal length twice (250 -> 125 -> 62), and global average pooling then turns each of the 128 feature maps into a single number.
2. **Timing branch**: a small dense network that turns the 3 RR-interval features into 16 numbers.

The 128 signal features and 16 timing features are joined into one vector of 144 numbers. A dense classifier then outputs one score for each of the 4 classes.

In [ ]:
class ResidualBlock1D(nn.Module):
    """Two convolutions whose result is added back to the block's input."""

    def __init__(self, channels, kernel_size=5):
        super().__init__()
        padding = kernel_size // 2    # keeps the signal length unchanged
        self.first_convolution = nn.Conv1d(channels, channels, kernel_size, padding=padding)
        self.first_batch_norm = nn.BatchNorm1d(channels)
        self.second_convolution = nn.Conv1d(channels, channels, kernel_size, padding=padding)
        self.second_batch_norm = nn.BatchNorm1d(channels)

    def forward(self, block_input):
        output = self.first_convolution(block_input)
        output = self.first_batch_norm(output)
        output = functional.relu(output)

        output = self.second_convolution(output)
        output = self.second_batch_norm(output)

        output = output + block_input      # the residual (skip) connection
        return functional.relu(output)


class ECGClassifier(nn.Module):
    def __init__(self, number_of_rr_features=3, number_of_classes=4, classifier_dropout=0.3):
        super().__init__()

        # ----- Signal branch -----
        self.input_layer = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
        )
        self.residual_block_1 = ResidualBlock1D(32)
        self.pooling_1 = nn.MaxPool1d(2)                  # length 250 -> 125
        self.dropout_1 = nn.Dropout(0.15)
        self.widen_channels_1 = nn.Conv1d(32, 64, kernel_size=1)

        self.residual_block_2 = ResidualBlock1D(64)
        self.pooling_2 = nn.MaxPool1d(2)                  # length 125 -> 62
        self.dropout_2 = nn.Dropout(0.15)
        self.widen_channels_2 = nn.Conv1d(64, 128, kernel_size=1)

        self.residual_block_3 = ResidualBlock1D(128)
        self.global_average_pooling = nn.AdaptiveAvgPool1d(1)

        # ----- Timing (RR interval) branch -----
        self.rr_feature_network = nn.Sequential(
            nn.Linear(number_of_rr_features, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
        )

        # ----- Final classifier: 128 signal features + 16 timing features -----
        self.classifier = nn.Sequential(
            nn.Linear(128 + 16, 64),
            nn.ReLU(),
            nn.Dropout(classifier_dropout),
            nn.Linear(64, number_of_classes),
        )

    def forward(self, signal, rr_features):
        # Signal branch
        signal_features = self.input_layer(signal)
        signal_features = self.residual_block_1(signal_features)
        signal_features = self.pooling_1(signal_features)
        signal_features = self.dropout_1(signal_features)
        signal_features = self.widen_channels_1(signal_features)

        signal_features = self.residual_block_2(signal_features)
        signal_features = self.pooling_2(signal_features)
        signal_features = self.dropout_2(signal_features)
        signal_features = self.widen_channels_2(signal_features)

        signal_features = self.residual_block_3(signal_features)
        signal_features = self.global_average_pooling(signal_features)
        signal_features = signal_features.squeeze(-1)     # (batch, 128, 1) -> (batch, 128)

        # Timing branch
        timing_features = self.rr_feature_network(rr_features)    # (batch, 16)

        # Join both branches and classify
        combined_features = torch.cat([signal_features, timing_features], dim=1)
        return self.classifier(combined_features)

## 5. Evaluation helpers and single-fold training

**Helper functions**
- `evaluate_model`: runs a model over labelled data and returns the macro F1 score, the predicted classes, the true classes and the class probabilities.
- `predict_probabilities`: runs a model over unlabelled data (the test set) and returns the class probabilities.

In [ ]:
def evaluate_model(model, data_loader):
    """Score a model on labelled data. Returns macro F1, predictions, true labels and probabilities."""
    model.eval()
    prediction_batches = []
    label_batches = []
    probability_batches = []

    with torch.no_grad():
        for signal, rr_features, labels in data_loader:
            signal = signal.to(device)
            rr_features = rr_features.to(device)

            scores = model(signal, rr_features)
            probabilities = functional.softmax(scores, dim=1).cpu().numpy()
            predicted_classes = probabilities.argmax(axis=1)

            probability_batches.append(probabilities)
            prediction_batches.append(predicted_classes)
            label_batches.append(labels.numpy())

    all_predictions = np.concatenate(prediction_batches)
    all_true_labels = np.concatenate(label_batches)
    all_probabilities = np.concatenate(probability_batches)

    macro_f1 = f1_score(all_true_labels, all_predictions, average="macro")
    return macro_f1, all_predictions, all_true_labels, all_probabilities


def predict_probabilities(model, data_loader):
    """Return class probabilities for unlabelled data (no true labels needed)."""
    model.eval()
    probability_batches = []

    with torch.no_grad():
        for signal, rr_features in data_loader:
            signal = signal.to(device)
            rr_features = rr_features.to(device)

            scores = model(signal, rr_features)
            probabilities = functional.softmax(scores, dim=1).cpu().numpy()
            probability_batches.append(probabilities)

    return np.concatenate(probability_batches)

**Training one fold**

`train_one_fold` trains a fresh model on one part of the data and validates it on the held-out part. Key design choices:

- **Scaling**: the timing features are standardised (mean 0, spread 1). The scaler is fitted on the training part only, so no information leaks from the validation part.
- **Class imbalance**: each beat gets a weight of `1 / sqrt(class size)`. These weights are used twice: in the sampler (rare classes are drawn more often) and in the loss (mistakes on rare classes cost more). The square root keeps the correction moderate, which is more stable than a full inverse-frequency weight.
- **Label smoothing** (0.05) makes the model slightly less over-confident.
- **Smoothed F1 for model selection**: validation F1 jumps around from epoch to epoch. Averaging the last `SMOOTHING_WINDOW` epochs gives a steadier signal for choosing the best weights, lowering the learning rate, and stopping early.
- **Early stopping**: training stops if the smoothed F1 does not improve for `EARLY_STOP_PATIENCE` epochs. The best weights are restored at the end.

In [ ]:
def train_one_fold(train_indices, validation_indices, fold_number):
    # --- Scale the timing features (fit on the training part only) ---
    rr_scaler = StandardScaler()
    train_rr_features = rr_scaler.fit_transform(all_rr_features[train_indices]).astype(np.float32)
    validation_rr_features = rr_scaler.transform(all_rr_features[validation_indices]).astype(np.float32)

    train_signals = all_signals[train_indices]
    validation_signals = all_signals[validation_indices]
    train_labels = all_labels[train_indices]
    validation_labels = all_labels[validation_indices]

    train_dataset = ECGDataset(train_signals, train_rr_features, train_labels, use_augmentation=True)
    validation_dataset = ECGDataset(validation_signals, validation_rr_features, validation_labels, use_augmentation=False)

    # --- Class weights: 1 / sqrt(number of beats in the class) ---
    class_counts = np.bincount(train_labels, minlength=NUMBER_OF_CLASSES).astype(np.float32)
    inverse_sqrt_frequency = 1.0 / np.sqrt(class_counts)

    # Sampler: rare classes are picked more often when building each training batch
    sample_weights = inverse_sqrt_frequency[train_labels]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

    # Loss: mistakes on rare classes cost more (weights add up to the number of classes)
    loss_weights = inverse_sqrt_frequency / inverse_sqrt_frequency.sum() * NUMBER_OF_CLASSES
    loss_weights_tensor = torch.tensor(loss_weights, dtype=torch.float32).to(device)
    print(f"[fold {fold_number}] class counts: {class_counts}, loss weights: {loss_weights}")

    train_loader = DataLoader(train_dataset, batch_size=256, sampler=sampler, num_workers=0)
    validation_loader = DataLoader(validation_dataset, batch_size=512, shuffle=False, num_workers=0)

    # --- Model, loss function, optimiser and learning-rate schedule ---
    model = ECGClassifier().to(device)
    loss_function = nn.CrossEntropyLoss(weight=loss_weights_tensor, label_smoothing=0.05)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    # Halve the learning rate when the smoothed F1 has not improved for 2 epochs
    learning_rate_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )

    validation_f1_history = []
    best_smoothed_f1 = -1
    best_model_weights = None
    epochs_without_improvement = 0
    start_time = time.time()

    for epoch in range(MAX_EPOCHS):
        # ----- Training pass -----
        model.train()
        total_loss = 0.0
        number_of_batches = 0

        for signal, rr_features, labels in train_loader:
            signal = signal.to(device)
            rr_features = rr_features.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            scores = model(signal, rr_features)
            loss = loss_function(scores, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)   # avoid huge updates
            optimizer.step()

            total_loss += loss.item()
            number_of_batches += 1

        # ----- Validation pass -----
        validation_f1, _, _, _ = evaluate_model(model, validation_loader)
        validation_f1_history.append(validation_f1)

        recent_f1_scores = validation_f1_history[-SMOOTHING_WINDOW:]
        smoothed_f1 = np.mean(recent_f1_scores)
        learning_rate_scheduler.step(smoothed_f1)

        average_loss = total_loss / number_of_batches
        elapsed_seconds = time.time() - start_time
        print(f"[fold {fold_number}] epoch {epoch + 1}/{MAX_EPOCHS} "
              f"loss={average_loss:.4f} val_f1={validation_f1:.4f} "
              f"smoothed_f1={smoothed_f1:.4f} time={elapsed_seconds:.1f}s")

        # ----- Keep the best weights, or stop early -----
        if smoothed_f1 > best_smoothed_f1:
            best_smoothed_f1 = smoothed_f1
            best_model_weights = {}
            for layer_name, layer_weights in model.state_dict().items():
                best_model_weights[layer_name] = layer_weights.clone()
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOP_PATIENCE:
                print(f"[fold {fold_number}] early stopping.")
                break

    # Go back to the best weights and score them once more on the validation part
    model.load_state_dict(best_model_weights)
    final_f1, final_predictions, _, final_probabilities = evaluate_model(model, validation_loader)
    print(f"[fold {fold_number}] best smoothed val macro F1: {best_smoothed_f1:.4f} "
          f"| final val macro F1: {final_f1:.4f}")

    return model, rr_scaler, final_f1, final_predictions, final_probabilities

## 6. Cross-validation training

The training data is split into 5 folds that keep the class proportions (`StratifiedKFold`). Each fold takes a turn as the validation set while a new model is trained on the other four.

Every training beat is therefore predicted exactly once by a model that never saw it. These **out-of-fold predictions** give an honest estimate of how well the approach works on new data.

This is the slowest cell in the notebook (roughly 30 minutes per fold on a GPU in the saved run).

In [ ]:
fold_splitter = StratifiedKFold(n_splits=NUMBER_OF_FOLDS, shuffle=True, random_state=RANDOM_SEED)

fold_models = []
fold_scalers = []
fold_f1_scores = []
out_of_fold_predictions = np.zeros(len(all_labels), dtype=np.int64)
out_of_fold_probabilities = np.zeros((len(all_labels), NUMBER_OF_CLASSES), dtype=np.float32)

fold_splits = fold_splitter.split(all_signals, all_labels)
for fold_number, (train_indices, validation_indices) in enumerate(fold_splits, start=1):
    model, rr_scaler, fold_f1, fold_predictions, fold_probabilities = train_one_fold(
        train_indices, validation_indices, fold_number
    )

    fold_models.append(model)
    fold_scalers.append(rr_scaler)
    fold_f1_scores.append(fold_f1)

    # Store each validation prediction at the position of its original row
    out_of_fold_predictions[validation_indices] = fold_predictions
    out_of_fold_probabilities[validation_indices] = fold_probabilities

formatted_fold_scores = []
for fold_f1 in fold_f1_scores:
    formatted_fold_scores.append(f"{fold_f1:.4f}")

print("\nPer-fold macro F1:", formatted_fold_scores)
print(f"Mean fold macro F1: {np.mean(fold_f1_scores):.4f} +/- {np.std(fold_f1_scores):.4f}")

## 7. Out-of-fold results

Here all out-of-fold predictions are scored together against the true labels:
- **Macro F1**: the competition metric.
- **Classification report**: precision, recall and F1 for each class.
- **Confusion matrix**: rows are the true classes and columns are the predicted classes. Numbers on the diagonal are correct predictions.

In the saved run, class 3 (the rarest) has high recall but low precision: the model finds most true class 3 beats but also labels some beats from other classes as class 3. Because Macro F1 weights all classes equally, this class has the largest effect on the final score.

In [ ]:
out_of_fold_macro_f1 = f1_score(all_labels, out_of_fold_predictions, average="macro")
print("Overall out-of-fold macro F1:", out_of_fold_macro_f1)
print(classification_report(all_labels, out_of_fold_predictions, digits=4))
print(confusion_matrix(all_labels, out_of_fold_predictions))

## 8. Save the trained models

The weights of all 5 models and the mean and scale of each fold's timing-feature scaler are stored in one file, so that predictions can be reproduced later without retraining.

In [ ]:
fold_model_weights = []
fold_scaler_means = []
fold_scaler_scales = []

for fold_index in range(len(fold_models)):
    fold_model_weights.append(fold_models[fold_index].state_dict())
    fold_scaler_means.append(fold_scalers[fold_index].mean_)
    fold_scaler_scales.append(fold_scalers[fold_index].scale_)

torch.save({
    "fold_states": fold_model_weights,
    "fold_scaler_means": fold_scaler_means,
    "fold_scaler_scales": fold_scaler_scales,
}, "model_kfold.pt")
print("Saved model_kfold.pt")

## 9. Predict the test set

The test data gets exactly the same preparation as the training data:
- missing timing values are filled with the **training** medians,
- signals are clipped to the range -10 to 10,
- timing features are scaled with each fold's own scaler.

Each of the 5 fold models then predicts class probabilities, and the probabilities are **averaged** (an ensemble). The class with the highest average probability is the final prediction for each beat.

In [ ]:
start_time = time.time()
test_data = pd.read_csv(f"{DATA_DIR}/test.csv")
load_seconds = time.time() - start_time
print("test shape:", test_data.shape, f"({load_seconds:.1f}s)")

test_signals = test_data[signal_columns].values.astype(np.float32)
test_rr_features = test_data[rr_interval_columns].values.astype(np.float32)

test_missing_count = np.isnan(test_rr_features).sum()
if test_missing_count > 0:
    print(f"Imputing {test_missing_count} NaN values in test extra features")
    test_rr_features = fill_missing_with_medians(test_rr_features, column_medians)   # training medians

test_signals = np.clip(test_signals, -10, 10)

# Add up each fold model's probabilities, divided by the number of models (= the average)
average_test_probabilities = np.zeros((len(test_data), NUMBER_OF_CLASSES), dtype=np.float32)
number_of_models = len(fold_models)

for fold_index in range(number_of_models):
    fold_model = fold_models[fold_index]
    fold_scaler = fold_scalers[fold_index]

    scaled_rr_features = fold_scaler.transform(test_rr_features).astype(np.float32)
    test_dataset = ECGDataset(test_signals, scaled_rr_features, labels=None, use_augmentation=False)
    test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False, num_workers=0)

    fold_probabilities = predict_probabilities(fold_model, test_loader)
    average_test_probabilities += fold_probabilities / number_of_models

test_predictions = average_test_probabilities.argmax(axis=1)
print("prediction distribution:", np.bincount(test_predictions, minlength=NUMBER_OF_CLASSES))

## 10. Write and check the submission file

The submission has two columns, `id` and `label`. The last cell checks it against `sample_submission.csv`: the number of rows, the ids (same values, same order), missing values, and the class counts.

In [ ]:
submission = pd.DataFrame({"id": test_data["id"], "label": test_predictions})
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
submission.head()

In [ ]:
sample_submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
print("submission shape:", submission.shape, "| sample shape:", sample_submission.shape)
print("ids match exactly:", (submission["id"].values == sample_submission["id"].values).all())
print("any nulls:", submission.isnull().sum().sum())
print("label value counts:\n", submission["label"].value_counts())